# Experimentos do grupo com o valor de cada item de frete

O grupo investigou duas perguntas complementares à previsão de tarifa (R$/t) do notebook 02. Quanto custa o item inteiro (R$)? Ele fica acima da mediana histórica? Este notebook refaz essas perguntas com a base tratada do projeto unificado e referências simples para comparação.

## Registro dos resultados originais

A versão publicada no [GitHub do grupo](https://github.com/gabrielribas18/analise-custos-frete/blob/0a7de43/analise/trabalho_agraria%20.ipynb) reporta LSTM com MAE de R$ 829,44, RMSE de R$ 1.345,00 e R² de 0,0838 em 18.352 previsões históricas. A sequência usou janelas de 10 registros e acompanhou o conjunto de teste durante o treino. Como o vínculo entre registros consecutivos de fretes diferentes não foi demonstrado, as métricas são exploratórias.

O [Colab compartilhado](https://colab.research.google.com/drive/1IrITaR7o22wmoCDTVV8TjNdfqDcOjxt1) registra os classificadores abaixo, avaliados em 18.362 registros. O SVM classificou todos os registros como de custo alto, o que ajuda a interpretar seu F1 da classe positiva.

| Classificador | Acurácia | F1 da classe de custo alto |
| --- | ---: | ---: |
| SVM | 0,5172 | 0,6818 |
| LightGBM | 0,7628 | 0,7689 |
| XGBoost | 0,7685 | 0,7723 |
| CatBoost | 0,7733 | 0,7748 |

As junções, a granularidade e as variáveis da base original diferem das usadas aqui. As métricas acima descrevem a execução do grupo e não permitem ranquear seus modelos frente aos deste notebook ou ao de tarifa.

## Base comum e divisão temporal

A preparação local mantém uma observação por número de frete e item. Execute `python scripts/build_data.py` na raiz caso precise gerar a base. Treinamos com os itens de 2025, testamos com os de 2026 e retiramos do treino qualquer número de frete presente no teste. A data de liquidação histórica é uma aproximação da data disponível no planejamento. A amostra inclui itens de valor zero e várias categorias de frete.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src/frete").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from frete.modeling import CATEGORICAL, NUMERIC, features, read_data, temporal_split

base = read_data(ROOT / "data/processed/base_modelagem_frete.csv")
target = "Valor líquido"
if base.duplicated(["Nº frete", "Item"]).any():
    raise ValueError("A base deve ter uma observação por frete e item.")

base = base.loc[
    base["ano"].isin([2025, 2026]) & np.isfinite(base[target])
].copy()
train, test, overlap = temporal_split(base, cutoff="2026-01-01")
if train.empty or test.empty:
    raise ValueError("São necessários itens com valor conhecido em 2025 e 2026.")
assert not set(train["Nº frete"]).intersection(test["Nº frete"])
print(f"Treino: {len(train):,} itens | Teste: {len(test):,} itens")
print(f"Números de frete em comum retirados do treino: {overlap}")

## Regressão direta do valor por item

O modelo de árvores com *gradient boosting* usa as variáveis básicas do modelo de tarifa, além da categoria do item (CtgI) registrada na base histórica. Valores realizados e indicadores calculados a partir do alvo ficam fora das entradas. As categorias são codificadas com o treino e categorias inéditas são tratadas como ausentes. A mediana do valor de treino é a referência. O WAPE divide a soma dos erros absolutos pela soma dos valores absolutos observados. Viés positivo indica superestimação no total.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

categorical = [*CATEGORICAL, "CtgI"]
x_train = features(train).assign(CtgI=train["CtgI"].fillna("NAO_INFORMADO"))
x_test = features(test).assign(CtgI=test["CtgI"].fillna("NAO_INFORMADO"))
y_train, y_test = train[target], test[target]
categorical_mask = [False] * len(NUMERIC) + [True] * len(categorical)

def boosting(estimator):
    preprocessor = ColumnTransformer([
        ("numericas", "passthrough", NUMERIC),
        ("categorias", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=np.nan,
            max_categories=255,
        ), categorical),
    ])
    return make_pipeline(preprocessor, estimator)

regressors = {
    "Mediana do treino": DummyRegressor(strategy="median"),
    "Gradient boosting": boosting(HistGradientBoostingRegressor(
        loss="absolute_error", categorical_features=categorical_mask,
        early_stopping=False, random_state=42,
    )),
}
regression_rows = []
for name, regressor in regressors.items():
    predicted = regressor.fit(x_train, y_train).predict(x_test)
    regression_rows.append({
        "modelo": name,
        "MAE (R$)": mean_absolute_error(y_test, predicted),
        "RMSE (R$)": root_mean_squared_error(y_test, predicted),
        "R²": r2_score(y_test, predicted),
        "WAPE (%)": 100 * np.abs(y_test - predicted).sum() / np.abs(y_test).sum(),
        "viés (%)": 100 * (predicted - y_test).sum() / y_test.sum(),
    })
display(pd.DataFrame(regression_rows).round(4))

## Classificação acima da mediana

O limiar vem somente do treino. Valores iguais ao limiar entram na classe de custo baixo. Comparamos *gradient boosting* com a estratégia de prever sempre a classe mais frequente no treino. A acurácia balanceada mede a média do *recall* das duas classes. A matriz de confusão mostra de qual classe vêm os erros.

In [ ]:
threshold = y_train.median()
class_train = y_train.gt(threshold).astype(int)
class_test = y_test.gt(threshold).astype(int)
if class_train.nunique() != 2 or class_test.nunique() != 2:
    raise ValueError("Treino e teste precisam conter as duas classes de custo.")
print(f"Mediana dos custos de 2025 no treino: R$ {threshold:.2f}")

classifiers = {
    "Classe majoritária no treino": DummyClassifier(strategy="most_frequent"),
    "Gradient boosting": boosting(HistGradientBoostingClassifier(
        categorical_features=categorical_mask, early_stopping=False,
        random_state=42,
    )),
}
classification_rows = []
confusion_tables = []
for name, classifier in classifiers.items():
    predicted = classifier.fit(x_train, class_train).predict(x_test)
    classification_rows.append({
        "modelo": name,
        "acurácia": accuracy_score(class_test, predicted),
        "acurácia balanceada": balanced_accuracy_score(class_test, predicted),
        "F1 alto": f1_score(class_test, predicted, zero_division=0),
        "F1 macro": f1_score(class_test, predicted, average="macro", zero_division=0),
    })
    confusion_tables.append(pd.DataFrame(
        confusion_matrix(class_test, predicted, labels=[0, 1]),
        index=pd.MultiIndex.from_product([[name], ["Real baixo", "Real alto"]]),
        columns=["Previsto baixo", "Previsto alto"],
    ))
display(pd.DataFrame(classification_rows).round(4))
display(pd.concat(confusion_tables))

## Aplicação dos resultados

A regressão estima o valor do item em reais e a classificação permite priorizar a revisão dos itens acima da mediana. A previsão de tarifa validada no notebook 02 continua sendo a referência do simulador. Estes experimentos usam configuração fixa e servem para medir desempenho histórico. A aplicação operacional exigiria verificar a disponibilidade das variáveis antes do embarque e testar em períodos posteriores.